# Analyze Library Usage in Agent-Authored PRs

This notebook analyzes how AI coding agents use external libraries in their pull requests.

Research Questions:
1. Do agents happily import and install new libraries?
2. Do they willingly use external libraries that are already installed or do they avoid it?
3. Do they try to commit invalid libraries?
4. Do they specify library versions in their PRs?
5. What are the most common libraries used by agents?

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import json
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.pr_analyzer import PRAnalyzer
from src.library_extractor import LibraryExtractor

tqdm.pandas()

data_dir = Path('../data')
output_dir = Path('../output')
output_dir.mkdir(exist_ok=True)

## Load Data

In [ ]:
# Load the datasets
print("Loading datasets...")
repo_df = pd.read_parquet(data_dir / 'repository.parquet')
pr_df = pd.read_parquet(data_dir / 'pull_request.parquet')
commit_details_df = pd.read_parquet(data_dir / 'pr_commit_details.parquet')

print(f"Repositories: {len(repo_df):,}")
print(f"Pull Requests: {len(pr_df):,}")
print(f"Commit Details: {len(commit_details_df):,}")

## Identify Top 3 Languages

In [ ]:
# Merge PR with repository to get language
pr_with_lang = pr_df.merge(
    repo_df[['repository_id', 'language']], 
    on='repository_id', 
    how='left'
)

# Count PRs by language
lang_counts = pr_with_lang['language'].value_counts()
print("Top 10 languages by PR count:")
print(lang_counts.head(10))

# Get top 3
top_3_langs = lang_counts.head(3).index.tolist()
print(f"\nTop 3 languages for analysis: {top_3_langs}")

## Filter Data to Top 3 Languages

In [ ]:
# Filter PRs to top 3 languages
top_lang_prs = pr_with_lang[pr_with_lang['language'].isin(top_3_langs)].copy()
print(f"PRs in top 3 languages: {len(top_lang_prs):,}")

# Get relevant commit details
pr_ids = set(top_lang_prs['pull_request_id'])
top_commits = commit_details_df[
    commit_details_df['pull_request_id'].isin(pr_ids)
].copy()
print(f"Commit details in top 3 languages: {len(top_commits):,}")

## Analyze Library Usage by Language

In [ ]:
analyzer = PRAnalyzer()
results_by_language = {}

for lang in top_3_langs:
    print(f"\n{'='*60}")
    print(f"Analyzing {lang}...")
    print('='*60)
    
    # Get PRs and commits for this language
    lang_pr_ids = set(top_lang_prs[top_lang_prs['language'] == lang]['pull_request_id'])
    lang_commits = top_commits[top_commits['pull_request_id'].isin(lang_pr_ids)]
    
    print(f"PRs: {len(lang_pr_ids):,}")
    print(f"File changes: {len(lang_commits):,}")
    
    # Analyze
    print(f"\nAnalyzing library usage...")
    results = analyzer.analyze_commit_details(lang_commits, lang)
    
    # Merge with agent info
    for pr_id, usage in results.items():
        pr_info = top_lang_prs[top_lang_prs['pull_request_id'] == pr_id]
        if len(pr_info) > 0 and 'agent' in pr_info.columns:
            usage.agent = pr_info.iloc[0]['agent']
    
    results_by_language[lang] = list(results.values())
    print(f"Analyzed {len(results):,} PRs")

## Generate Statistics

In [ ]:
stats_by_language = {}

for lang, usages in results_by_language.items():
    print(f"\n{'='*60}")
    print(f"{lang} Statistics")
    print('='*60)
    
    stats = PRAnalyzer.aggregate_statistics(usages)
    stats_by_language[lang] = stats
    
    print(f"\nTotal PRs: {stats['total_prs']:,}")
    print(f"PRs with new libraries: {stats['prs_with_new_libs']:,} ({stats['pct_prs_with_new_libs']:.1f}%)")
    print(f"PRs with dependency changes: {stats['prs_with_dep_changes']:,} ({stats['pct_prs_with_dep_changes']:.1f}%)")
    print(f"PRs adding dep file: {stats['prs_adding_dep_file']:,}")
    print(f"PRs modifying dep file: {stats['prs_modifying_dep_file']:,}")
    
    print(f"\nUnique libraries: {stats['total_unique_libs']:,}")
    print(f"External libraries: {stats['total_external_libs']:,}")
    print(f"Stdlib imports: {stats['total_stdlib_imports']:,}")
    
    print(f"\nAvg libraries per PR: {stats['avg_libs_per_pr']:.2f}")
    print(f"Avg new libraries per PR: {stats['avg_new_libs_per_pr']:.2f}")
    
    print(f"\nLibraries with version: {stats['total_libs_with_version']:,}")
    print(f"Libraries without version: {stats['total_libs_without_version']:,}")
    print(f"% with version: {stats['pct_libs_with_version']:.1f}%")
    
    if stats['version_operators']:
        print(f"\nVersion operators:")
        for op, count in stats['version_operators'].most_common():
            print(f"  {op}: {count}")
    
    print(f"\nMost common libraries:")
    for lib, count in stats['most_common_libs'][:10]:
        print(f"  {lib}: {count}")
    
    print(f"\nMost common new libraries:")
    for lib, count in stats['most_common_new_libs'][:10]:
        print(f"  {lib}: {count}")

## Visualizations

In [ ]:
# Library usage comparison across languages
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. PRs with new libraries
langs = list(stats_by_language.keys())
pct_with_new = [stats_by_language[l]['pct_prs_with_new_libs'] for l in langs]
axes[0, 0].bar(langs, pct_with_new)
axes[0, 0].set_title('% of PRs Adding New Libraries')
axes[0, 0].set_ylabel('Percentage')

# 2. Avg libraries per PR
avg_libs = [stats_by_language[l]['avg_libs_per_pr'] for l in langs]
axes[0, 1].bar(langs, avg_libs)
axes[0, 1].set_title('Average Libraries per PR')
axes[0, 1].set_ylabel('Count')

# 3. % with version specs
pct_versioned = [stats_by_language[l]['pct_libs_with_version'] for l in langs]
axes[1, 0].bar(langs, pct_versioned)
axes[1, 0].set_title('% of Dependencies with Version Specs')
axes[1, 0].set_ylabel('Percentage')

# 4. External vs Stdlib ratio
external_counts = [stats_by_language[l]['total_external_libs'] for l in langs]
stdlib_counts = [stats_by_language[l]['total_stdlib_imports'] for l in langs]
x = range(len(langs))
width = 0.35
axes[1, 1].bar([i - width/2 for i in x], external_counts, width, label='External')
axes[1, 1].bar([i + width/2 for i in x], stdlib_counts, width, label='Stdlib')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(langs)
axes[1, 1].set_title('External vs Stdlib Libraries')
axes[1, 1].set_ylabel('Unique Count')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(output_dir / 'library_usage_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Save Results

In [ ]:
# Save detailed results
for lang, usages in results_by_language.items():
    # Convert to list of dicts
    results_list = [u.to_dict() for u in usages]
    
    # Save as JSON
    output_file = output_dir / f'{lang.lower()}_library_usage.json'
    with open(output_file, 'w') as f:
        json.dump(results_list, f, indent=2)
    print(f"Saved {lang} results to {output_file}")

# Save aggregated statistics
stats_file = output_dir / 'aggregated_statistics.json'
# Convert Counter objects to dicts for JSON serialization
stats_serializable = {}
for lang, stats in stats_by_language.items():
    stats_copy = stats.copy()
    if 'version_operators' in stats_copy:
        stats_copy['version_operators'] = dict(stats_copy['version_operators'])
    stats_serializable[lang] = stats_copy

with open(stats_file, 'w') as f:
    json.dump(stats_serializable, f, indent=2)
print(f"\nSaved aggregated statistics to {stats_file}")

## Summary of Findings

This section will be updated after running the analysis with key insights about:

1. **Library Adoption**: Do agents readily add new libraries?
2. **Existing Libraries**: Do agents use already-installed libraries or avoid dependencies?
3. **Version Specifications**: How often do agents specify versions?
4. **Most Common Libraries**: What are the go-to libraries for each language?
5. **Invalid Libraries**: Pattern detection for potentially invalid dependencies